# D392 — Snowflake table versions and recovery

This notebook teaches a complete but deliberately small lifecycle with **one permanent ecommerce table**: `PRODUCT_CATALOG_VERSIONED`. The only helper table is a session-scoped temporary table used as the source of one `MERGE`.

Run the SQL blocks from top to bottom in the **same Snowsight worksheet session**. Session variables such as `$BAD_UPDATE_ID` and the temporary table disappear when that session ends.

## 1. What a table version means in Snowflake

Snowflake stores table data in immutable micro-partitions. An `INSERT`, `UPDATE`, `DELETE`, `MERGE`, or `TRUNCATE` writes new data/metadata rather than editing old micro-partitions in place. At commit, the new state becomes current; older state remains available while Time Travel retention covers it.

Snowflake does not expose a user-facing sequential table version number like Delta Lake. A historical point is selected with a timestamp, a negative offset, or a statement query ID.

| Need | Snowflake mechanism |
| --- | --- |
| Cancel work before commit | `ROLLBACK` |
| Read an already committed older state | `AT` or `BEFORE` |
| Restore rows after a committed mistake | Read history and write it forward |
| Restore a dropped table | `UNDROP TABLE` |
| Control accessible history | `DATA_RETENTION_TIME_IN_DAYS` |
| Vacuum old native-table files | Snowflake manages expiry; there is no user-run `VACUUM` for a native table |

## 2. Lab setup

Change the role or warehouse names if your account uses different conventions. `ERROR_ON_NONDETERMINISTIC_MERGE = TRUE` protects the target when multiple source rows could update the same target row.

```sql
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS D39_SNOWFLAKE_COMPLETE;
CREATE SCHEMA IF NOT EXISTS D39_SNOWFLAKE_COMPLETE.TABLE_VERSIONS;
CREATE WAREHOUSE IF NOT EXISTS D39_LEARNING_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE D39_LEARNING_WH;
USE SCHEMA D39_SNOWFLAKE_COMPLETE.TABLE_VERSIONS;
ALTER SESSION SET TIMEZONE = 'UTC';
ALTER SESSION SET AUTOCOMMIT = TRUE;
ALTER SESSION SET ERROR_ON_NONDETERMINISTIC_MERGE = TRUE;
```

## 3. Create and seed the one permanent table

The table begins with one day of Time Travel retention, which works on Standard Edition. `SOURCE_VERSION` is a business version supplied by an upstream system; it is separate from Snowflake's internal historical states.

The first two statements intentionally reset this lab if it is rerun. Do not use a reset step against a real production table.

```sql
DROP TABLE IF EXISTS PRODUCT_CATALOG_VERSIONED;

CREATE TABLE PRODUCT_CATALOG_VERSIONED (
  PRODUCT_ID      NUMBER         NOT NULL,
  SKU             VARCHAR        NOT NULL,
  PRODUCT_NAME    VARCHAR        NOT NULL,
  CATEGORY        VARCHAR        NOT NULL,
  PRICE           NUMBER(10, 2)  NOT NULL,
  STOCK_QTY       NUMBER         NOT NULL,
  STATUS          VARCHAR        NOT NULL,
  SOURCE_VERSION  NUMBER         NOT NULL,
  UPDATED_AT      TIMESTAMP_LTZ  NOT NULL
)
DATA_RETENTION_TIME_IN_DAYS = 1
COMMENT = 'Single-table ecommerce lifecycle lab';

INSERT INTO PRODUCT_CATALOG_VERSIONED
  (PRODUCT_ID, SKU, PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY, STATUS, SOURCE_VERSION, UPDATED_AT)
VALUES
  (101, 'ELEC-KEY-01', 'Compact Keyboard',  'Electronics', 49.90,  25, 'ACTIVE', 1, CURRENT_TIMESTAMP()),
  (102, 'ELEC-MSE-01', 'Wireless Mouse',    'Electronics', 29.90,  40, 'ACTIVE', 1, CURRENT_TIMESTAMP()),
  (103, 'HOME-LMP-01', 'Desk Lamp',         'Home',        39.00,  18, 'ACTIVE', 1, CURRENT_TIMESTAMP()),
  (104, 'HOME-MUG-01', 'Stoneware Mug',     'Home',        14.50,  60, 'ACTIVE', 1, CURRENT_TIMESTAMP());

SELECT *
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

Expected result: four active products.

## 4. Ordinary DML creates committed states

Snowflake normally has `AUTOCOMMIT = TRUE`, so each standalone DML statement is its own transaction. These three statements create three committed points in history.

```sql
INSERT INTO PRODUCT_CATALOG_VERSIONED
  (PRODUCT_ID, SKU, PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY, STATUS, SOURCE_VERSION, UPDATED_AT)
VALUES
  (105, 'SPORT-BTL-01', 'Steel Bottle', 'Sports', 24.00, 35, 'ACTIVE', 1, CURRENT_TIMESTAMP());

UPDATE PRODUCT_CATALOG_VERSIONED
SET PRICE = 27.50,
    SOURCE_VERSION = 2,
    UPDATED_AT = CURRENT_TIMESTAMP()
WHERE PRODUCT_ID = 105;

DELETE FROM PRODUCT_CATALOG_VERSIONED
WHERE PRODUCT_ID = 104;

SELECT *
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

Expected result: product 105 exists at 27.50 and product 104 is absent.

## 5. ACID in this example

| Property | What to observe |
| --- | --- |
| Atomicity | Both statements commit together, or `ROLLBACK` cancels both. A failed DML statement is itself atomic. |
| Consistency | `NOT NULL` and numeric definitions reject invalid rows; business rules still belong in validated application/ELT logic. |
| Isolation | Other transactions do not see this transaction's uncommitted changes. |
| Durability | After `COMMIT`, the new state persists; Time Travel can still expose older retained states. |

### Commit two changes as one unit

```sql
BEGIN TRANSACTION;

UPDATE PRODUCT_CATALOG_VERSIONED
SET PRICE = 47.90, SOURCE_VERSION = 2, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE PRODUCT_ID = 101;

UPDATE PRODUCT_CATALOG_VERSIONED
SET STOCK_QTY = STOCK_QTY - 2, SOURCE_VERSION = 2, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE PRODUCT_ID = 103;

COMMIT;

SELECT PRODUCT_ID, PRICE, STOCK_QTY, SOURCE_VERSION
FROM PRODUCT_CATALOG_VERSIONED
WHERE PRODUCT_ID IN (101, 103)
ORDER BY PRODUCT_ID;
```

The two row changes become durable at the same transaction boundary.

## 6. Roll back an uncommitted mistake

The middle `SELECT` runs inside the transaction and shows the bad values. `ROLLBACK` then discards both changes.

```sql
BEGIN TRANSACTION;

UPDATE PRODUCT_CATALOG_VERSIONED
SET PRICE = 0, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE PRODUCT_ID = 102;

DELETE FROM PRODUCT_CATALOG_VERSIONED
WHERE PRODUCT_ID = 103;

-- Visible to this transaction before rollback.
SELECT PRODUCT_ID, PRICE, STOCK_QTY
FROM PRODUCT_CATALOG_VERSIONED
WHERE PRODUCT_ID IN (102, 103)
ORDER BY PRODUCT_ID;

ROLLBACK;

-- Product 102 has its old price and product 103 exists again.
SELECT PRODUCT_ID, PRICE, STOCK_QTY
FROM PRODUCT_CATALOG_VERSIONED
WHERE PRODUCT_ID IN (102, 103)
ORDER BY PRODUCT_ID;
```

A DDL statement such as `ALTER TABLE` implicitly commits an active transaction before executing as its own transaction. Keep DDL out of a DML transaction when rollback behavior matters. If an application sees an unexpected statement error inside an explicit transaction, issue `ROLLBACK` explicitly rather than assuming the whole transaction was cancelled.

## 7. Merge a temporary change set

The target remains the same permanent table. The temporary table exists only in this session and is used to stage inserts, updates, and deletes. Two updates arrive for product 102; `QUALIFY` selects the newest source version so the `MERGE` is deterministic.

```sql
CREATE OR REPLACE TEMP TABLE PRODUCT_CHANGES_TEMP (
  CHANGE_ID       NUMBER,
  PRODUCT_ID      NUMBER,
  SKU             VARCHAR,
  PRODUCT_NAME    VARCHAR,
  CATEGORY        VARCHAR,
  PRICE           NUMBER(10, 2),
  STOCK_QTY       NUMBER,
  STATUS          VARCHAR,
  SOURCE_VERSION  NUMBER,
  OPERATION       VARCHAR,
  EVENT_TS        TIMESTAMP_LTZ
);

INSERT INTO PRODUCT_CHANGES_TEMP VALUES
  (1, 102, 'ELEC-MSE-01', 'Wireless Mouse', 'Electronics', 27.90, 50, 'ACTIVE', 3, 'UPSERT', CURRENT_TIMESTAMP()),
  (2, 102, 'ELEC-MSE-01', 'Wireless Mouse', 'Electronics', 26.90, 55, 'ACTIVE', 4, 'UPSERT', CURRENT_TIMESTAMP()),
  (3, 103, 'HOME-LMP-01', 'Desk Lamp',      'Home',        39.00, 16, 'ACTIVE', 3, 'DELETE', CURRENT_TIMESTAMP()),
  (4, 106, 'OFFC-STD-01', 'Laptop Stand',   'Office',      58.00, 20, 'ACTIVE', 1, 'UPSERT', CURRENT_TIMESTAMP());

MERGE INTO PRODUCT_CATALOG_VERSIONED AS T
USING (
  SELECT PRODUCT_ID, SKU, PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY,
         STATUS, SOURCE_VERSION, OPERATION, EVENT_TS
  FROM PRODUCT_CHANGES_TEMP
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY PRODUCT_ID
    ORDER BY SOURCE_VERSION DESC, CHANGE_ID DESC
  ) = 1
) AS S
ON T.PRODUCT_ID = S.PRODUCT_ID
WHEN MATCHED AND S.OPERATION = 'DELETE'
                 AND S.SOURCE_VERSION > T.SOURCE_VERSION THEN
  DELETE
WHEN MATCHED AND S.OPERATION = 'UPSERT'
                 AND S.SOURCE_VERSION > T.SOURCE_VERSION THEN
  UPDATE SET
    T.SKU = S.SKU,
    T.PRODUCT_NAME = S.PRODUCT_NAME,
    T.CATEGORY = S.CATEGORY,
    T.PRICE = S.PRICE,
    T.STOCK_QTY = S.STOCK_QTY,
    T.STATUS = S.STATUS,
    T.SOURCE_VERSION = S.SOURCE_VERSION,
    T.UPDATED_AT = S.EVENT_TS
WHEN NOT MATCHED AND S.OPERATION = 'UPSERT' THEN
  INSERT (PRODUCT_ID, SKU, PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY, STATUS, SOURCE_VERSION, UPDATED_AT)
  VALUES (S.PRODUCT_ID, S.SKU, S.PRODUCT_NAME, S.CATEGORY, S.PRICE, S.STOCK_QTY, S.STATUS, S.SOURCE_VERSION, S.EVENT_TS);

SELECT *
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

Expected result: product 102 is version 4, product 103 is deleted, and product 106 is inserted. Re-running the same merge makes no target changes because the source versions are not newer.

## 8. Create a committed incident and capture its query ID

This mistake is a standalone statement, so it commits immediately. `ROLLBACK` is now too late. Capture `LAST_QUERY_ID()` immediately after the damaging statement; any intervening statement would change what it returns.

```sql
UPDATE PRODUCT_CATALOG_VERSIONED
SET PRICE = PRICE * 10,
    UPDATED_AT = CURRENT_TIMESTAMP()
WHERE CATEGORY = 'Electronics';

SET BAD_UPDATE_ID = LAST_QUERY_ID();

SELECT $BAD_UPDATE_ID AS BAD_UPDATE_QUERY_ID;

SELECT PRODUCT_ID, PRODUCT_NAME, PRICE
FROM PRODUCT_CATALOG_VERSIONED
WHERE CATEGORY = 'Electronics'
ORDER BY PRODUCT_ID;
```

Expected result: the electronics prices are ten times too large. Keep the displayed query ID for incident notes; the session variable is only a convenience.

## 9. Query current and historical states

`BEFORE (STATEMENT => id)` returns the point immediately before that statement. `AT (STATEMENT => id)` is inclusive of the statement. A statement query ID must be no more than 14 days old, and the required table data must still be inside its Time Travel retention window.

```sql
SELECT 'CURRENT_BAD_STATE' AS SNAPSHOT, PRODUCT_ID, PRODUCT_NAME, PRICE
FROM PRODUCT_CATALOG_VERSIONED
WHERE CATEGORY = 'Electronics'

UNION ALL

SELECT 'BEFORE_BAD_UPDATE' AS SNAPSHOT, PRODUCT_ID, PRODUCT_NAME, PRICE
FROM PRODUCT_CATALOG_VERSIONED
  BEFORE (STATEMENT => $BAD_UPDATE_ID)
WHERE CATEGORY = 'Electronics'

ORDER BY SNAPSHOT, PRODUCT_ID;
```

The older rows are read-only. Time Travel does not move the current table pointer backward.

Other point selectors are shown below. Run the offset query only after this table has existed for at least 60 seconds. Replace the timestamp literal with a time after table creation and inside retention.

```sql
-- Exactly 60 seconds before the query begins. OFFSET must be a negative constant expression.
-- SELECT * FROM PRODUCT_CATALOG_VERSIONED AT (OFFSET => -60);

-- A typed timestamp avoids session-format ambiguity.
-- SELECT *
-- FROM PRODUCT_CATALOG_VERSIONED
--   AT (TIMESTAMP => TO_TIMESTAMP_LTZ('2026-09-08 10:30:00 +00:00'));

-- This includes the bad UPDATE and therefore shows the bad prices.
SELECT PRODUCT_ID, PRODUCT_NAME, PRICE
FROM PRODUCT_CATALOG_VERSIONED
  AT (STATEMENT => $BAD_UPDATE_ID)
WHERE CATEGORY = 'Electronics'
ORDER BY PRODUCT_ID;
```

## 10. Recover by writing the old values forward

The historical version becomes the `MERGE` source and the same table remains the target. This creates a new committed current state; it does not erase the incident from retained history. The unmatched branch also restores a row if the damaging statement deleted it.

```sql
MERGE INTO PRODUCT_CATALOG_VERSIONED AS T
USING (
  SELECT *
  FROM PRODUCT_CATALOG_VERSIONED
    BEFORE (STATEMENT => $BAD_UPDATE_ID)
) AS H
ON T.PRODUCT_ID = H.PRODUCT_ID
WHEN MATCHED THEN
  UPDATE SET
    T.SKU = H.SKU,
    T.PRODUCT_NAME = H.PRODUCT_NAME,
    T.CATEGORY = H.CATEGORY,
    T.PRICE = H.PRICE,
    T.STOCK_QTY = H.STOCK_QTY,
    T.STATUS = H.STATUS,
    T.SOURCE_VERSION = H.SOURCE_VERSION,
    T.UPDATED_AT = H.UPDATED_AT
WHEN NOT MATCHED THEN
  INSERT (PRODUCT_ID, SKU, PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY, STATUS, SOURCE_VERSION, UPDATED_AT)
  VALUES (H.PRODUCT_ID, H.SKU, H.PRODUCT_NAME, H.CATEGORY, H.PRICE, H.STOCK_QTY, H.STATUS, H.SOURCE_VERSION, H.UPDATED_AT);

SET RESTORE_QUERY_ID = LAST_QUERY_ID();

SELECT PRODUCT_ID, PRODUCT_NAME, CATEGORY, PRICE
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

The catalog is correct again. To delete rows that were newly inserted by a broader incident, first compare current keys to the historical key set, review the result, and then delete those extra keys in the same explicit recovery transaction. This lab's incident only updated existing rows.

## 11. Prove the incident still exists in history

Current rows are repaired, but `AT` the bad update still returns the incident state. `AT` the restore returns the corrected state.

```sql
SELECT 'AT_BAD_UPDATE' AS SNAPSHOT, PRODUCT_ID, PRICE
FROM PRODUCT_CATALOG_VERSIONED AT (STATEMENT => $BAD_UPDATE_ID)
WHERE CATEGORY = 'Electronics'

UNION ALL

SELECT 'AT_RESTORE' AS SNAPSHOT, PRODUCT_ID, PRICE
FROM PRODUCT_CATALOG_VERSIONED AT (STATEMENT => $RESTORE_QUERY_ID)
WHERE CATEGORY = 'Electronics'

ORDER BY SNAPSHOT, PRODUCT_ID;
```

## 12. Schema evolution on the same table

Schema evolution changes the current table definition while retained historical row data remains available. Historical queries use the **current schema**, so after a rename use `PRODUCT_TITLE`, and a column that did not exist at the historical point is returned as `NULL`.

Use the already captured restore-statement ID as a precise point under the old schema, then add a column, populate it, rename a column, and widen the price precision.

```sql
ALTER TABLE PRODUCT_CATALOG_VERSIONED ADD COLUMN BRAND VARCHAR;

UPDATE PRODUCT_CATALOG_VERSIONED
SET BRAND = CASE CATEGORY
  WHEN 'Electronics' THEN 'Northstar'
  WHEN 'Sports'     THEN 'Trailcraft'
  WHEN 'Office'     THEN 'Workwell'
  ELSE 'Homeworks'
END;

ALTER TABLE PRODUCT_CATALOG_VERSIONED
  RENAME COLUMN PRODUCT_NAME TO PRODUCT_TITLE;

ALTER TABLE PRODUCT_CATALOG_VERSIONED
  ALTER COLUMN PRICE SET DATA TYPE NUMBER(12, 2);

DESCRIBE TABLE PRODUCT_CATALOG_VERSIONED;

SELECT PRODUCT_ID, PRODUCT_TITLE, BRAND, PRICE
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

Now query a point captured before those DDL changes, using the current column names.

```sql
SELECT PRODUCT_ID, PRODUCT_TITLE, BRAND, PRICE
FROM PRODUCT_CATALOG_VERSIONED
  AT (STATEMENT => $RESTORE_QUERY_ID)
ORDER BY PRODUCT_ID;
```

Expected result: the historical product values are present, `PRODUCT_TITLE` is the valid current name, and `BRAND` is `NULL` at that older point. Dropping a column removes it from the current schema, so it can no longer be selected through a historical query even while its old storage bytes remain retained.

## 13. Retention and Fail-safe

Inspect the effective table setting, then set the lab-safe one-day value explicitly.

```sql
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS'
  IN TABLE PRODUCT_CATALOG_VERSIONED;

ALTER TABLE PRODUCT_CATALOG_VERSIONED
  SET DATA_RETENTION_TIME_IN_DAYS = 1;

SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS'
  IN TABLE PRODUCT_CATALOG_VERSIONED;
```

| Table kind / edition | Time Travel retention | After Time Travel |
| --- | --- | --- |
| Permanent, Standard Edition | `0` or `1` day | 7-day Fail-safe |
| Permanent, Enterprise or higher | `0` through `90` days | 7-day Fail-safe |
| Transient or temporary | `0` or `1` day | No Fail-safe |

A value of `0` effectively disables Time Travel. Fail-safe is a Snowflake-operated recovery period, not an interactive query window and not a substitute for Time Travel. Lowering retention can make older history inaccessible as it ages out.

These examples are intentionally comments:

```sql
-- Enterprise Edition or higher, permanent table only:
-- ALTER TABLE PRODUCT_CATALOG_VERSIONED SET DATA_RETENTION_TIME_IN_DAYS = 7;

-- Do not run during this lab; it disables the recovery examples:
-- ALTER TABLE PRODUCT_CATALOG_VERSIONED SET DATA_RETENTION_TIME_IN_DAYS = 0;
```

## 14. Recover from `TRUNCATE`

`TRUNCATE TABLE` removes every current row but preserves the table object. Capture its query ID and insert the rows from immediately before it. The schema is unchanged, so the column order matches.

```sql
TRUNCATE TABLE PRODUCT_CATALOG_VERSIONED;
SET TRUNCATE_QUERY_ID = LAST_QUERY_ID();

SELECT COUNT(*) AS CURRENT_ROWS_AFTER_TRUNCATE
FROM PRODUCT_CATALOG_VERSIONED;

SELECT COUNT(*) AS ROWS_BEFORE_TRUNCATE
FROM PRODUCT_CATALOG_VERSIONED
  BEFORE (STATEMENT => $TRUNCATE_QUERY_ID);

INSERT INTO PRODUCT_CATALOG_VERSIONED
SELECT *
FROM PRODUCT_CATALOG_VERSIONED
  BEFORE (STATEMENT => $TRUNCATE_QUERY_ID);

SELECT *
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

Expected result: zero current rows immediately after `TRUNCATE`, then the catalog is restored by a new `INSERT`. In production, list columns explicitly if source and target schemas might differ.

## 15. Recover a dropped table with `UNDROP`

A dropped permanent table can be restored while it is still in Time Travel. The table name must be free. `SHOW TABLES HISTORY` includes retained dropped versions.

```sql
DROP TABLE PRODUCT_CATALOG_VERSIONED;

SHOW TABLES HISTORY LIKE 'PRODUCT_CATALOG_VERSIONED'
  IN SCHEMA D39_SNOWFLAKE_COMPLETE.TABLE_VERSIONS;

UNDROP TABLE PRODUCT_CATALOG_VERSIONED;

SELECT *
FROM PRODUCT_CATALOG_VERSIONED
ORDER BY PRODUCT_ID;
```

If a new table already uses the same name, rename that active table before `UNDROP`. If several dropped versions share a name, inspect `SHOW TABLES HISTORY`; modern Snowflake also supports `UNDROP TABLE IDENTIFIER(<table_id>)` to choose a specific retained version.

## 16. Inspect metadata and storage states

Snowflake exposes table metadata and retained-byte categories, but it does not provide a Delta-style `DESCRIBE HISTORY` sequence of every row version. Query history and the IDs captured by the application form the operational audit trail.

```sql
SHOW TABLES HISTORY LIKE 'PRODUCT_CATALOG_VERSIONED'
  IN SCHEMA D39_SNOWFLAKE_COMPLETE.TABLE_VERSIONS;

SELECT GET_DDL('TABLE', 'PRODUCT_CATALOG_VERSIONED');

SELECT
  TABLE_NAME,
  IS_TRANSIENT,
  ACTIVE_BYTES,
  TIME_TRAVEL_BYTES,
  FAILSAFE_BYTES,
  TABLE_CREATED,
  TABLE_DROPPED
FROM D39_SNOWFLAKE_COMPLETE.INFORMATION_SCHEMA.TABLE_STORAGE_METRICS
WHERE TABLE_SCHEMA = 'TABLE_VERSIONS'
  AND TABLE_NAME = 'PRODUCT_CATALOG_VERSIONED'
ORDER BY TABLE_CREATED DESC;
```

Storage counters can take time to reflect tiny lab changes. `ACTIVE_BYTES` belongs to current data, `TIME_TRAVEL_BYTES` covers retained historical data, and `FAILSAFE_BYTES` covers permanent-table data that has left Time Travel. `CREATE OR REPLACE TABLE` is logically a drop plus create and can leave another retained table version, which is one reason to use it carefully on long-lived tables.

## 17. Decision guide

| Situation | Correct response |
| --- | --- |
| Bad changes are uncommitted | `ROLLBACK` the transaction. |
| A committed update/delete/merge was wrong | Validate `BEFORE` history, then write the old rows forward in a controlled transaction. |
| All rows were truncated | Insert from `BEFORE (STATEMENT => truncate_query_id)`. |
| The table was dropped | `UNDROP TABLE` inside retention. |
| A reproducible investigation point is needed | Record the DML query ID; use `AT`/`BEFORE (STATEMENT => ...)`. |
| More recovery time is needed | Increase retention before the incident; permanent tables need Enterprise Edition or higher for more than one day. |
| Old versions should expire | Let Snowflake enforce retention and Fail-safe; do not look for a native-table `VACUUM`. |

A safe recovery workflow is: stop writers, identify the exact bad statement or timestamp, compare current and historical rows, test the recovery query as a `SELECT`, apply it in an explicit transaction, validate counts and values, then commit.

## 18. Optional cleanup

Run this only when you no longer need the lab. The temporary table also disappears automatically when the session ends.

```sql
DROP TABLE IF EXISTS PRODUCT_CHANGES_TEMP;
DROP TABLE IF EXISTS PRODUCT_CATALOG_VERSIONED;

-- Optional shared-object cleanup:
-- DROP SCHEMA IF EXISTS D39_SNOWFLAKE_COMPLETE.TABLE_VERSIONS;
-- DROP WAREHOUSE IF EXISTS D39_LEARNING_WH;
-- DROP DATABASE IF EXISTS D39_SNOWFLAKE_COMPLETE;
```

## Official references

- [Transactions](https://docs.snowflake.com/en/sql-reference/transactions)
- [Understanding and using Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel)
- [`AT | BEFORE`](https://docs.snowflake.com/en/sql-reference/constructs/at-before)
- [`MERGE`](https://docs.snowflake.com/en/sql-reference/sql/merge)
- [`UNDROP TABLE`](https://docs.snowflake.com/en/sql-reference/sql/undrop-table)
- [`DATA_RETENTION_TIME_IN_DAYS`](https://docs.snowflake.com/en/sql-reference/parameters#data-retention-time-in-days)
- [`TABLE_STORAGE_METRICS`](https://docs.snowflake.com/en/sql-reference/info-schema/table_storage_metrics)